In [ ]:
# 导入 LangChain 1.0 核心模块（注意模块路径变化）
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain.agents import create_react_agent  # 1.0 智能体创建函数
from langchain.agents import AgentExecutor  # 1.0 智能体执行器（核心）
from langchain import hub  # 1.0 原生 hub（替代 langchain_classic.hub）
import os
import dotenv

# 1. 读取环境变量（逻辑不变，确保密钥正确加载）
dotenv.load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY1")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL")
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

# 2. 初始化工具（1.0 中无需用 StructuredTool 包装，直接使用即可）
# TavilySearchResults 已默认实现 Tool 接口，可直接加入工具列表
search_tool = TavilySearchResults(
    max_results=10,  # 保留旧版的「最多返回10条结果」配置
    name="Search",  # 工具名称（可选，默认已为"TavilySearchResults"，可自定义）
    description="用于检索互联网实时/最新知识（如时事、数据、动态信息等）"  # 工具描述（优化清晰度）
)
tools = [search_tool]  # 智能体可调用的工具列表

# 3. 初始化 LLM（配置不变，1.0 兼容旧版 LLM 初始化逻辑）
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.8,  # 保留中等随机性，让智能体决策更灵活
    max_tokens=200  # 限制智能体单次输出长度，避免冗余
)

# 4. 从 LangChain Hub 获取 1.0 兼容的智能体 Prompt（关键：替代旧版 classic hub）
# 使用主流的 "react-chat" 模板（适合对话式智能体，支持工具调用逻辑）
agent_prompt = hub.pull("hwchase17/react-chat")

# 5. 创建智能体（1.0 用 create_react_agent，需传入 LLM、工具、Prompt）
agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=agent_prompt
)

# 6. 创建智能体执行器（1.0 新增：负责调度智能体的「思考-调用工具-生成结果」流程）
agent_executor = AgentExecutor.from_agent_and_tools(
    agent=agent,
    tools=tools,
    verbose=True  # 开启详细日志（可选，便于调试：查看智能体是否调用工具、调用逻辑）
)

# 7. 运行智能体（1.0 用 invoke 方法，传入用户查询）
result = agent_executor.invoke({
    "input": "2024年全球AI领域融资总额是多少？"  # 示例：需实时数据，智能体会自动调用搜索工具
})

# 打印最终结果
print("智能体输出：", result["output"])